# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, zero-shot LOKO validation, baselines, MOO, figures.

**Run cells in order.**

In [ ]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

In [ ]:
# Cell 2: Pull latest code and install dependencies
!git pull
!uv sync

In [ ]:
# Cell 3: Monotonicity ground truth audit (Paper Section 4.1)
!uv run python cli.py analysis monotonicity --max-groups 5000

In [ ]:
# Cell 4: Train HINN (Random Split mode)
!uv run python cli.py train train --epochs 500 --batch-size 1024 --seed 42

## Leave-One-Kernel-Out (LOKO) Validation
This is the primary scientific novelty: testing zero-shot generalization on unseen hardware.

In [ ]:
# Cell 5: LOKO Sweep (example: hold out stencil2d)
!uv run python cli.py train train --epochs 500 --loko stencil2d --seed 42

In [ ]:
# Cell 6: Train baselines (XGBoost + Vanilla MLP)
!uv run python cli.py train baselines --epochs 500 --seed 42

In [ ]:
# Cell 7: Multi-Objective Optimization
!uv run python cli.py moo run

In [ ]:
# Cell 8: Generate Figures
!uv run python cli.py plot training-dynamics
!uv run python cli.py plot pareto
!uv run python cli.py plot monotonicity
!uv run python cli.py plot comparison

## Push Results to GitHub
Requires `GITHUB_PAT` to be set in Colab Secrets (left sidebar, key icon).

In [ ]:
# Cell 9: Authenticated Push
import os
from google.colab import userdata

try:
    pat = userdata.get('GITHUB_PAT')
    repo_url = f"https://{pat}@github.com/sattary/2601_chip_paper.git"
    
    !git config --global user.email "thesattary@gmail.com"
    !git config --global user.name "sat"
    !git add results/ models/  # Only add results and model weights
    !git commit -m "HINN training results from Colab sync"
    !git remote set-url origin {repo_url}
    !git push origin master
    print("Results successfully pushed to GitHub!")
except Exception as e:
    print(f"Sync failed: {e}")
    print("Check that GITHUB_PAT is added to Colab Secrets.")